<a href="https://colab.research.google.com/github/vitor-michelucci/postech-tech-challenge-fase2-template/blob/main/notebooks/00_preparacao_dados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 00 — Preparação dos Dados

Este notebook realiza a preparação inicial dos dados utilizados no Tech Challenge — Fase 2.

## Objetivos

- Obter os dados originais a partir da fonte pública.
- Verificar a estrutura das bases disponíveis.
- Analisar a relação entre os identificadores das bases.
- Estudar o histórico de crédito necessário para definição da variável-alvo.
- Construir e justificar a variável-alvo.
- Consolidar as informações em uma base analítica para as etapas seguintes.

> **Fonte dos dados:** Credit Card Approval Prediction, disponibilizado no Kaggle.

In [1]:
# ============================================================
# 1. CONFIGURAÇÃO DO AMBIENTE
# ============================================================

from pathlib import Path

# Endereço do repositório do projeto
REPO_URL = "https://github.com/vitor-michelucci/postech-tech-challenge-fase2-template.git"

# Local onde o repositório ficará disponível no Colab
PROJECT_PATH = Path("/content/postech-tech-challenge-fase2-template")

# Clona o repositório somente se ele ainda não estiver disponível
if not PROJECT_PATH.exists():
    !git clone {REPO_URL}
else:
    print("Repositório já disponível nesta sessão.")

print("\nDiretório do projeto:")
print(PROJECT_PATH)

Cloning into 'postech-tech-challenge-fase2-template'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 41 (delta 5), reused 3 (delta 3), pack-reused 30 (from 1)
Receiving objects: 100% (41/41), 20.62 KiB | 1.03 MiB/s, done.
Resolving deltas: 100% (5/5), done.

Diretório do projeto:
/content/postech-tech-challenge-fase2-template


In [2]:
# ============================================================
# 2. OBTENÇÃO DOS DADOS ORIGINAIS
# ============================================================

import shutil
import kagglehub

# Baixa o dataset ou utiliza a versão já disponível no cache
dataset_path = kagglehub.dataset_download(
    "rikdifos/credit-card-approval-prediction"
)

KAGGLE_PATH = Path(dataset_path)

# Pasta destinada aos dados originais dentro do projeto
RAW_PATH = PROJECT_PATH / "data" / "raw"

# Garante que a pasta exista
RAW_PATH.mkdir(parents=True, exist_ok=True)

# Arquivos originais necessários para o projeto
arquivos = [
    "application_record.csv",
    "credit_record.csv"
]

# Copia os arquivos para data/raw
for nome_arquivo in arquivos:
    origem = KAGGLE_PATH / nome_arquivo
    destino = RAW_PATH / nome_arquivo

    if not destino.exists():
        shutil.copy2(origem, destino)
        print(f"Copiado: {nome_arquivo}")
    else:
        print(f"Já disponível: {nome_arquivo}")

print("\nDados originais disponíveis em:")
print(RAW_PATH)

100%|██████████| 5.32M/5.32M [00:00<00:00, 143MB/s]

Extracting files...


Copiado: application_record.csv
Copiado: credit_record.csv

Dados originais disponíveis em:
/content/postech-tech-challenge-fase2-template/data/raw


In [3]:
# ============================================================
# 3. VERIFICAÇÃO DOS ARQUIVOS ORIGINAIS
# ============================================================

for nome_arquivo in arquivos:
    arquivo = RAW_PATH / nome_arquivo

    tamanho_mb = arquivo.stat().st_size / (1024 ** 2)

    print(f"{nome_arquivo}: {tamanho_mb:.2f} MB")

application_record.csv: 51.83 MB
credit_record.csv: 14.66 MB


## 2 — Conhecimento inicial das bases

Nesta etapa, as duas bases originais são carregadas separadamente para análise de sua estrutura, dimensões, variáveis e tipos de dados.

Nenhuma transformação é realizada neste momento.

In [4]:
# ============================================================
# 4. CARREGAMENTO DAS BASES
# ============================================================

import pandas as pd

# Carrega os arquivos originais
df_application = pd.read_csv(
    RAW_PATH / "application_record.csv"
)

df_credit = pd.read_csv(
    RAW_PATH / "credit_record.csv"
)

print("Bases carregadas com sucesso.")

Bases carregadas com sucesso.


In [5]:
# Dimensões das bases
print("application_record:", df_application.shape)
print("credit_record:", df_credit.shape)

application_record: (438557, 18)
credit_record: (1048575, 3)


In [6]:
# ============================================================
# 5. COLUNAS DISPONÍVEIS EM CADA BASE
# ============================================================

print("Colunas de application_record:")
print(df_application.columns.tolist())

print("\nColunas de credit_record:")
print(df_credit.columns.tolist())

Colunas de application_record:
['ID', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'CNT_CHILDREN', 'AMT_INCOME_TOTAL', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'DAYS_BIRTH', 'DAYS_EMPLOYED', 'FLAG_MOBIL', 'FLAG_WORK_PHONE', 'FLAG_PHONE', 'FLAG_EMAIL', 'OCCUPATION_TYPE', 'CNT_FAM_MEMBERS']

Colunas de credit_record:
['ID', 'MONTHS_BALANCE', 'STATUS']


In [7]:
# ============================================================
# 6. ANÁLISE DOS IDENTIFICADORES
# ============================================================

print("application_record")
print("Linhas:", len(df_application))
print("IDs únicos:", df_application["ID"].nunique())

print("\ncredit_record")
print("Linhas:", len(df_credit))
print("IDs únicos:", df_credit["ID"].nunique())

application_record
Linhas: 438557
IDs únicos: 438510

credit_record
Linhas: 1048575
IDs únicos: 45985


In [8]:
# ============================================================
# 7. RELAÇÃO ENTRE OS IDs DAS DUAS BASES
# ============================================================

# Conjuntos de IDs únicos de cada base
ids_application = set(df_application["ID"])
ids_credit = set(df_credit["ID"])

# IDs presentes nas duas bases
ids_comuns = ids_application.intersection(ids_credit)

print("IDs únicos em application_record:", len(ids_application))
print("IDs únicos em credit_record:", len(ids_credit))
print("IDs presentes nas duas bases:", len(ids_comuns))

IDs únicos em application_record: 438510
IDs únicos em credit_record: 45985
IDs presentes nas duas bases: 36457


In [9]:
# ============================================================
# 8. INVESTIGAÇÃO DE IDs DUPLICADOS EM application_record
# ============================================================

# Identifica todas as linhas cujo ID aparece mais de uma vez
duplicados_application = df_application[
    df_application["ID"].duplicated(keep=False)
].sort_values("ID")

print("Quantidade de linhas com IDs duplicados:",
      len(duplicados_application))

print("Quantidade de IDs envolvidos:",
      duplicados_application["ID"].nunique())

Quantidade de linhas com IDs duplicados: 94
Quantidade de IDs envolvidos: 47


In [10]:
# ============================================================
# 9. VERIFICAÇÃO DOS REGISTROS COM IDs DUPLICADOS
# ============================================================

# Exibe os registros dos primeiros IDs duplicados
duplicados_application.head(10)

,ID,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,DAYS_BIRTH,DAYS_EMPLOYED,FLAG_MOBIL,FLAG_WORK_PHONE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS
425023,7022197,F,N,Y,0,450000.0,Commercial associate,Higher education,Separated,House / apartment,-19813,-1799,1,0,0,1,NaN,1.0
426818,7022197,M,Y,Y,3,135000.0,Working,Secondary / secondary special,Married,House / apartment,-11945,-735,1,0,0,1,Laborers,5.0
431911,7022327,M,Y,Y,0,256500.0,Commercial associate,Higher education,Married,House / apartment,-21503,-1674,1,0,0,1,Core staff,2.0
431545,7022327,F,N,Y,0,135000.0,Commercial associate,Secondary / secondary special,Single / not married,House / apartment,-14771,-5298,1,0,0,0,High skill tech staff,1.0
425486,7023108,M,Y,Y,1,67500.0,Working,Secondary / secondary special,Married,House / apartment,-15156,-1696,1,1,0,0,Core staff,3.0
426488,7023108,F,N,N,0,135000.0,Working,Secondary / secondary special,Married,House / apartment,-17590,-1273,1,0,0,0,Cleaning staff,2.0
425306,7023651,F,N,N,0,225000.0,Commercial associate,Incomplete higher,Single / not married,House / apartment,-10229,-1209,1,0,0,0,Accountants,1.0
421907,7023651,M,Y,N,1,157500.0,Commercial associate,Incomplete higher,Married,House / apartment,-10521,-1457,1,0,0,0,Drivers,3.0
427778,7024111,M,N,N,2,157500.0,Commercial associate,Secondary / secondary special,Civil marriage,House / apartment,-15270,-117,1,1,0,0,Drivers,4.0
432643,7024111,F,N,Y,0,180000.0,Working,Secondary / secondary special,Married,House / apartment,-22041,-1524,1,1,1,0,Accountants,2.0


In [11]:
# ============================================================
# 10. IDs DUPLICADOS QUE POSSUEM HISTÓRICO DE CRÉDITO
# ============================================================

# IDs que aparecem mais de uma vez na base cadastral
ids_duplicados = set(duplicados_application["ID"])

# Verifica quais deles também aparecem na base de crédito
ids_duplicados_com_credito = ids_duplicados.intersection(ids_credit)

print("IDs duplicados em application_record:",
      len(ids_duplicados))

print("IDs duplicados que também aparecem em credit_record:",
      len(ids_duplicados_com_credito))

IDs duplicados em application_record: 47
IDs duplicados que também aparecem em credit_record: 0


### Análise dos identificadores

A base `application_record` possui 438.557 registros e 438.510 IDs únicos. Foram identificados 47 IDs repetidos, cada um associado a dois registros cadastrais distintos.

A inspeção desses casos mostrou que não se tratam de duplicatas exatas, pois existem diferenças entre as características cadastrais associadas ao mesmo identificador.

Entretanto, nenhum dos 47 IDs duplicados está presente na base `credit_record`. Dessa forma, esses registros não pertencem ao conjunto de clientes para os quais existe histórico de crédito e, consequentemente, não integrarão a base analítica utilizada para construção da variável-alvo e posterior modelagem.

As bases possuem 36.457 IDs em comum.

## 3 — Análise do histórico de crédito

A base `credit_record` contém registros históricos associados aos clientes. Antes da definição da variável-alvo, serão analisadas as variáveis `MONTHS_BALANCE` e `STATUS`, buscando compreender sua estrutura e distribuição.

A definição do target será realizada somente após essa análise.

In [12]:
# ============================================================
# 11. PRIMEIROS REGISTROS DO HISTÓRICO DE CRÉDITO
# ============================================================

df_credit.head(10)

,ID,MONTHS_BALANCE,STATUS
0,5001711,0,X
1,5001711,-1,0
2,5001711,-2,0
3,5001711,-3,0
4,5001712,0,C
5,5001712,-1,C
6,5001712,-2,C
7,5001712,-3,C
8,5001712,-4,C
9,5001712,-5,C


In [13]:
# ============================================================
# 12. DISTRIBUIÇÃO DA VARIÁVEL STATUS
# ============================================================

# Contagem de registros para cada situação de crédito
status_counts = (
    df_credit["STATUS"]
    .value_counts()
    .sort_index()
)

print(status_counts)

STATUS
0    383120
1     11090
2       868
3       320
4       223
5      1693
C    442031
X    209230
Name: count, dtype: int64


In [14]:
# Percentual de cada categoria de STATUS
status_percentual = (
    df_credit["STATUS"]
    .value_counts(normalize=True)
    .mul(100)
    .sort_index()
    .round(2)
)

print(status_percentual)

STATUS
0    36.54
1     1.06
2     0.08
3     0.03
4     0.02
5     0.16
C    42.16
X    19.95
Name: proportion, dtype: float64


In [15]:
# ============================================================
# 13. QUANTIDADE DE MESES OBSERVADOS POR CLIENTE
# ============================================================

meses_por_cliente = (
    df_credit
    .groupby("ID")
    .size()
)

print(meses_por_cliente.describe())

count    45985.000000
mean        22.802544
std         15.492771
min          1.000000
25%         10.000000
50%         19.000000
75%         34.000000
max         61.000000
dtype: float64


In [16]:
# ============================================================
# 14. INTERVALO TEMPORAL DO HISTÓRICO
# ============================================================

print("Maior valor de MONTHS_BALANCE:",
      df_credit["MONTHS_BALANCE"].max())

print("Menor valor de MONTHS_BALANCE:",
      df_credit["MONTHS_BALANCE"].min())

print("Quantidade de valores distintos:",
      df_credit["MONTHS_BALANCE"].nunique())

Maior valor de MONTHS_BALANCE: 0
Menor valor de MONTHS_BALANCE: -60
Quantidade de valores distintos: 61


In [17]:
# ============================================================
# 15. PIOR NÍVEL DE ATRASO OBSERVADO POR CLIENTE
# ============================================================

# Converte apenas os status de atraso para valores numéricos.
# C e X não representam faixas de atraso e, por isso,
# ficam como valores ausentes nesta análise.
mapa_status = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5
}

df_credit["STATUS_NUM"] = df_credit["STATUS"].map(mapa_status)

# Identifica o maior nível de atraso observado para cada cliente
pior_status_cliente = (
    df_credit
    .groupby("ID")["STATUS_NUM"]
    .max()
)

print(pior_status_cliente.value_counts(dropna=False).sort_index())

STATUS_NUM
0.0    34682
1.0     4683
2.0      336
3.0       88
4.0       48
5.0      195
NaN     5953
Name: count, dtype: int64


In [18]:
# ============================================================
# 16. CLIENTES POR DIFERENTES NÍVEIS DE ATRASO
# ============================================================

total_clientes = df_credit["ID"].nunique()

for limite in [1, 2, 3, 4, 5]:

    quantidade = (pior_status_cliente >= limite).sum()
    percentual = quantidade / total_clientes * 100

    print(
        f"STATUS >= {limite}: "
        f"{quantidade} clientes "
        f"({percentual:.2f}%)"
    )

STATUS >= 1: 5350 clientes (11.63%)
STATUS >= 2: 667 clientes (1.45%)
STATUS >= 3: 331 clientes (0.72%)
STATUS >= 4: 243 clientes (0.53%)
STATUS >= 5: 195 clientes (0.42%)


In [19]:
# ============================================================
# 17. NÍVEIS DE ATRASO NO UNIVERSO UTILIZÁVEL PARA MODELAGEM
# ============================================================

# Mantém somente clientes presentes nas duas bases
pior_status_comum = pior_status_cliente[
    pior_status_cliente.index.isin(ids_comuns)
]

total_comum = len(pior_status_comum)

print("Clientes presentes nas duas bases:", total_comum)
print()

for limite in [1, 2, 3, 4, 5]:

    quantidade = (pior_status_comum >= limite).sum()
    percentual = quantidade / total_comum * 100

    print(
        f"STATUS >= {limite}: "
        f"{quantidade} clientes "
        f"({percentual:.2f}%)"
    )

Clientes presentes nas duas bases: 36457

STATUS >= 1: 4291 clientes (11.77%)
STATUS >= 2: 616 clientes (1.69%)
STATUS >= 3: 302 clientes (0.83%)
STATUS >= 4: 226 clientes (0.62%)
STATUS >= 5: 180 clientes (0.49%)


In [20]:
# ============================================================
# 18. CLIENTES SEM STATUS NUMÉRICO
# ============================================================

# IDs comuns cujo histórico não possui nenhum STATUS de 0 a 5
ids_sem_status_numerico = pior_status_comum[
    pior_status_comum.isna()
].index

print(
    "Clientes sem STATUS de 0 a 5:",
    len(ids_sem_status_numerico)
)

print(
    "Percentual:",
    f"{len(ids_sem_status_numerico) / total_comum * 100:.2f}%"
)

Clientes sem STATUS de 0 a 5: 4455
Percentual: 12.22%


In [21]:
# Filtra o histórico desses clientes
historico_sem_status = df_credit[
    df_credit["ID"].isin(ids_sem_status_numerico)
]

# Verifica quais STATUS aparecem
print(
    historico_sem_status["STATUS"]
    .value_counts()
)

STATUS
X    75005
C    18660
Name: count, dtype: int64


In [22]:
# Identifica a combinação de STATUS existente para cada cliente
padroes_status = (
    historico_sem_status
    .groupby("ID")["STATUS"]
    .apply(lambda x: tuple(sorted(set(x))))
    .value_counts()
)

print(padroes_status)

STATUS
(X,)      3347
(C, X)     806
(C,)       302
Name: count, dtype: int64


In [23]:
# ============================================================
# 19. CLIENTES COM HISTÓRICO DE CRÉDITO OBSERVÁVEL
# ============================================================

clientes_com_status_numerico = pior_status_comum.notna().sum()

print(
    "Clientes com pelo menos um STATUS entre 0 e 5:",
    clientes_com_status_numerico
)

print(
    "Percentual:",
    f"{clientes_com_status_numerico / total_comum * 100:.2f}%"
)

Clientes com pelo menos um STATUS entre 0 e 5: 32002
Percentual: 87.78%


In [24]:
# ============================================================
# 20. PERFIL DO HISTÓRICO DISPONÍVEL POR CLIENTE
# ============================================================

# Trabalha somente com os clientes presentes nas duas bases
credito_comum = df_credit[
    df_credit["ID"].isin(ids_comuns)
].copy()

# Para cada cliente, identifica os tipos de informação existentes
perfil_historico = (
    credito_comum
    .groupby("ID")["STATUS"]
    .agg(
        tem_status_numerico=lambda x: x.isin(
            ["0", "1", "2", "3", "4", "5"]
        ).any(),
        tem_C=lambda x: (x == "C").any(),
        tem_X=lambda x: (x == "X").any()
    )
)

# Conta as combinações encontradas
resumo_perfis = (
    perfil_historico
    .value_counts()
    .reset_index(name="quantidade")
)

resumo_perfis

,tem_status_numerico,tem_C,tem_X,quantidade
0,True,True,False,11457
1,True,False,True,10110
2,True,True,True,5385
3,True,False,False,5050
4,False,False,True,3347
5,False,True,True,806
6,False,True,False,302


### Síntese preliminar do histórico de crédito

Entre os 36.457 clientes presentes nas duas bases, 32.002 (87,78%) possuem pelo menos um registro de `STATUS` entre `0` e `5`, permitindo observar diretamente algum nível de comportamento de pagamento ou atraso.

Os 4.455 clientes restantes possuem exclusivamente registros `C` e/ou `X`. Destes, 3.347 apresentam somente `X`, 302 somente `C` e 806 uma combinação de `C` e `X`.

Esses grupos não serão classificados automaticamente como bons ou maus clientes. A definição da população elegível e da variável-alvo será realizada considerando o significado dos status, o horizonte de observação e o impacto das diferentes regras sobre o balanceamento das classes.

In [25]:
# ============================================================
# 21. TABELA DIAGNÓSTICA PARA DEFINIÇÃO DO TARGET
# ============================================================

# Uma linha para cada cliente presente nas duas bases
diagnostico_target = perfil_historico.copy()

# Acrescenta o pior status numérico observado
diagnostico_target["pior_status"] = pior_status_comum

# Marca provisoriamente a ocorrência de atraso >= 30 dias
# Esta ainda NÃO é a variável-alvo definitiva.
diagnostico_target["evento_30d"] = (
    diagnostico_target["pior_status"] >= 1
)

diagnostico_target.head(10)

,tem_status_numerico,tem_C,tem_X,pior_status,evento_30d
ID,,,,,
5008804,True,True,True,1.0,True
5008805,True,True,True,1.0,True
5008806,True,True,True,0.0,False
5008808,True,False,True,0.0,False
5008809,False,False,True,NaN,False
5008810,True,True,True,0.0,False
5008811,True,True,True,0.0,False
5008812,True,False,True,0.0,False
5008813,True,False,True,0.0,False


In [26]:
# Quantidade de clientes segundo presença de histórico
# numérico e ocorrência do evento de 30+ dias

resumo_evento = (
    diagnostico_target
    .groupby(
        ["tem_status_numerico", "evento_30d"],
        dropna=False
    )
    .size()
    .reset_index(name="quantidade")
)

resumo_evento

,tem_status_numerico,evento_30d,quantidade
0,False,False,4455
1,True,False,27711
2,True,True,4291


In [27]:
# ============================================================
# 22. SIMULAÇÃO DA REGRA PROPOSTA PARA O TARGET
# ============================================================

import numpy as np

# Começa sem atribuir classe a nenhum cliente
diagnostico_target["target_proposto"] = np.nan

# TARGET = 1
# Cliente apresentou atraso de 30 dias ou mais
diagnostico_target.loc[
    diagnostico_target["pior_status"] >= 1,
    "target_proposto"
] = 1

# TARGET = 0
# Cliente teve status 0, mas nunca chegou a 30 dias de atraso
diagnostico_target.loc[
    diagnostico_target["pior_status"] == 0,
    "target_proposto"
] = 0

# TARGET = 0
# Cliente não possui status numérico,
# mas possui pelo menos um registro C
diagnostico_target.loc[
    diagnostico_target["pior_status"].isna()
    & diagnostico_target["tem_C"],
    "target_proposto"
] = 0

In [28]:
# Distribuição da regra proposta

print(
    diagnostico_target["target_proposto"]
    .value_counts(dropna=False)
)

print("\nPercentuais:")

print(
    diagnostico_target["target_proposto"]
    .value_counts(normalize=True, dropna=False)
    .mul(100)
    .round(2)
)

target_proposto
0.0    28819
1.0     4291
NaN     3347
Name: count, dtype: int64

Percentuais:
target_proposto
0.0    79.05
1.0    11.77
NaN     9.18
Name: proportion, dtype: float64


## 4 — Definição da variável-alvo

A variável-alvo foi definida a partir do pior comportamento de crédito observado para cada cliente ao longo do histórico disponível.

Foram considerados como **evento (`target = 1`)** os clientes que apresentaram pelo menos um registro de atraso igual ou superior a 30 dias (`STATUS >= 1`).

Foram considerados como **não evento (`target = 0`)**:

- clientes cujo pior registro foi `STATUS = 0`, correspondente a atraso entre 1 e 29 dias, sem ocorrência posterior de atraso igual ou superior a 30 dias;
- clientes sem status numérico, mas com pelo menos um registro `C`, indicando crédito quitado no período observado.

Clientes que apresentaram exclusivamente registros `X` foram mantidos sem classificação e excluídos da população utilizada na modelagem, pois a ausência de empréstimo no período não fornece evidência suficiente para classificá-los como bons ou maus pagadores.

A escolha do ponto de corte de 30 dias considerou simultaneamente a interpretação do comportamento de atraso e a distribuição observada na base. Cortes mais severos produziriam classes de evento extremamente reduzidas: 1,69% dos clientes atingiram 60 dias ou mais de atraso e 0,83% atingiram 90 dias ou mais, enquanto o critério de 30 dias ou mais identificou 11,77% dos clientes presentes nas duas bases.

Após a aplicação da regra, 33.110 clientes possuem target definido, sendo 4.291 eventos e 28.819 não eventos.

In [29]:
# ============================================================
# 23. CONSOLIDAÇÃO DA VARIÁVEL-ALVO
# ============================================================

# Mantém somente clientes para os quais foi possível
# definir a variável-alvo
df_target = (
    diagnostico_target
    .dropna(subset=["target_proposto"])
    [["target_proposto"]]
    .rename(columns={"target_proposto": "target"})
    .copy()
)

# Converte o target para inteiro: 0 ou 1
df_target["target"] = df_target["target"].astype(int)

print("Clientes com target definido:", len(df_target))

print("\nDistribuição:")
print(df_target["target"].value_counts())

print("\nPercentual:")
print(
    df_target["target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Clientes com target definido: 33110

Distribuição:
target
0    28819
1     4291
Name: count, dtype: int64

Percentual:
target
0    87.04
1    12.96
Name: proportion, dtype: float64


In [30]:
# ============================================================
# 24. VALIDAÇÃO DA VARIÁVEL-ALVO
# ============================================================

print("Número de linhas:", len(df_target))
print("Número de IDs únicos:", df_target.index.nunique())
print("Targets ausentes:", df_target["target"].isna().sum())
print("Valores possíveis:", sorted(df_target["target"].unique()))

Número de linhas: 33110
Número de IDs únicos: 33110
Targets ausentes: 0
Valores possíveis: [np.int64(0), np.int64(1)]


## 5 — Construção da base analítica

A base analítica é construída pela associação entre as características cadastrais de `application_record` e a variável-alvo derivada do histórico de crédito.

As variáveis utilizadas diretamente na construção do target, como `STATUS`, `MONTHS_BALANCE` e o pior nível de atraso observado, não são incorporadas como variáveis explicativas, evitando vazamento de informação (*data leakage*).

A junção é realizada pelo identificador `ID`, mantendo somente os clientes para os quais foi possível definir a variável-alvo.

In [31]:
# ============================================================
# 25. CONSTRUÇÃO DA BASE ANALÍTICA
# ============================================================

# Transforma o índice de df_target novamente em coluna ID
target_merge = df_target.reset_index()

# Associa as características cadastrais ao target
df_analitico = df_application.merge(
    target_merge,
    on="ID",
    how="inner"
)

print("Dimensão da base analítica:")
print(df_analitico.shape)

Dimensão da base analítica:
(33110, 19)


In [32]:
# ============================================================
# 26. VALIDAÇÃO DA BASE ANALÍTICA
# ============================================================

print("Número de linhas:", len(df_analitico))
print("Número de IDs únicos:", df_analitico["ID"].nunique())
print("IDs duplicados:", df_analitico["ID"].duplicated().sum())
print("Targets ausentes:", df_analitico["target"].isna().sum())

print("\nDistribuição do target:")
print(df_analitico["target"].value_counts())

print("\nPercentual do target:")
print(
    df_analitico["target"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

Número de linhas: 33110
Número de IDs únicos: 33110
IDs duplicados: 0
Targets ausentes: 0

Distribuição do target:
target
0    28819
1     4291
Name: count, dtype: int64

Percentual do target:
target
0    87.04
1    12.96
Name: proportion, dtype: float64


### Validação da base analítica

Após a junção, a base analítica contém 33.110 clientes distintos e 19 variáveis, incluindo o target. Não foram identificados IDs duplicados nem valores ausentes na variável-alvo.

Antes da persistência da base, são verificadas sua estrutura e a presença de valores ausentes. O tratamento desses valores será realizado posteriormente, na etapa de pré-processamento.

In [33]:
# ============================================================
# 27. ESTRUTURA DA BASE ANALÍTICA
# ============================================================

df_analitico.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 33110 entries, 0 to 33109
Data columns (total 19 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   ID                   33110 non-null  int64  
 1   CODE_GENDER          33110 non-null  object 
 2   FLAG_OWN_CAR         33110 non-null  object 
 3   FLAG_OWN_REALTY      33110 non-null  object 
 4   CNT_CHILDREN         33110 non-null  int64  
 5   AMT_INCOME_TOTAL     33110 non-null  float64
 6   NAME_INCOME_TYPE     33110 non-null  object 
 7   NAME_EDUCATION_TYPE  33110 non-null  object 
 8   NAME_FAMILY_STATUS   33110 non-null  object 
 9   NAME_HOUSING_TYPE    33110 non-null  object 
 10  DAYS_BIRTH           33110 non-null  int64  
 11  DAYS_EMPLOYED        33110 non-null  int64  
 12  FLAG_MOBIL           33110 non-null  int64  
 13  FLAG_WORK_PHONE      33110 non-null  int64  
 14  FLAG_PHONE           33110 non-null  int64  
 15  FLAG_EMAIL           33110 non-null 

In [34]:
# ============================================================
# 28. VALORES AUSENTES
# ============================================================

ausentes = (
    df_analitico
    .isna()
    .sum()
    .sort_values(ascending=False)
)

# Exibe somente variáveis que possuem valores ausentes
ausentes[ausentes > 0]

,0
OCCUPATION_TYPE,10373


### Base analítica final

A inspeção identificou valores ausentes apenas na variável `OCCUPATION_TYPE`. Esses valores são mantidos nesta etapa, pois seu tratamento será definido posteriormente no notebook de pré-processamento.

A base resultante deste notebook representa, portanto, a base analítica inicial do projeto, contendo as características cadastrais dos clientes e a variável-alvo construída a partir do histórico de crédito.

In [35]:
# ============================================================
# 29. SALVAMENTO DA BASE ANALÍTICA
# ============================================================

PROCESSED_PATH = PROJECT_PATH / "data" / "processed"
PROCESSED_PATH.mkdir(parents=True, exist_ok=True)

arquivo_saida = PROCESSED_PATH / "dataset_base.csv"

df_analitico.to_csv(
    arquivo_saida,
    index=False
)

print("Base analítica salva com sucesso.")
print("Arquivo:", arquivo_saida)

Base analítica salva com sucesso.
Arquivo: /content/postech-tech-challenge-fase2-template/data/processed/dataset_base.csv


In [36]:
# ============================================================
# 30. VALIDAÇÃO DO ARQUIVO GERADO
# ============================================================

# Recarrega a base diretamente do arquivo salvo
df_validacao = pd.read_csv(arquivo_saida)

print("Dimensão do arquivo:", df_validacao.shape)
print("IDs únicos:", df_validacao["ID"].nunique())
print("IDs duplicados:", df_validacao["ID"].duplicated().sum())
print("Targets ausentes:", df_validacao["target"].isna().sum())

print("\nDistribuição do target:")
print(df_validacao["target"].value_counts())

print("\nArquivo:")
print(arquivo_saida)

Dimensão do arquivo: (33110, 19)
IDs únicos: 33110
IDs duplicados: 0
Targets ausentes: 0

Distribuição do target:
target
0    28819
1     4291
Name: count, dtype: int64

Arquivo:
/content/postech-tech-challenge-fase2-template/data/processed/dataset_base.csv


## 6 — Conclusão

A etapa de preparação resultou em uma base analítica com **33.110 clientes e 19 variáveis**, incluindo a variável-alvo.

A variável-alvo foi construída a partir do histórico de crédito, considerando como evento a ocorrência de atraso igual ou superior a 30 dias. Clientes com histórico exclusivamente `X` foram excluídos da população de modelagem por não apresentarem evidência suficiente de comportamento de pagamento.

A base final apresenta:

- 28.819 clientes classificados como não evento (87,04%);
- 4.291 clientes classificados como evento (12,96%);
- nenhum ID duplicado;
- nenhum valor ausente na variável-alvo.

Foi identificada ausência de informação em `OCCUPATION_TYPE`, cujo tratamento será avaliado posteriormente na etapa de pré-processamento.

A base analítica foi armazenada em:

`data/processed/dataset_base.csv`

Esse arquivo será utilizado como ponto de partida para a análise exploratória dos dados.

### Limitação

O período de histórico disponível não é uniforme entre os clientes. Dessa forma, clientes com maior número de meses observados possuem maior oportunidade de apresentar o evento definido para a variável-alvo. Essa característica deve ser considerada como uma limitação metodológica da construção do target.